<a href="https://colab.research.google.com/github/A1phar1u5/FuzzyLSTM_With_Indicators_Test_System/blob/main/FuzzyLSTM_With_Indicators_Test_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Импортируем необходимые библиотеки
import tensorflow as tf
from keras.layers import Dense, Activation, Dropout, Layer, RNN
from keras.layers import LSTM, Input
from keras.models import Sequential
from tensorflow.keras.layers import LSTMCell
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from keras.initializers import Constant
from keras.utils import get_custom_objects
from tensorflow.keras.layers import Input, RNN, Dense, Lambda, Activation

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score, mean_absolute_percentage_error

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import date,timedelta

from prettytable import PrettyTable
import matplotlib as mpl
import matplotlib.pyplot as plt

from google.colab import drive

import os, csv, time, json, random, math, itertools, hashlib

from typing import Dict, Any




plt.style.use("grayscale")  #ч/б палитра
mpl.rcParams["figure.dpi"] = 140
mpl.rcParams["savefig.dpi"] = 140
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.linestyle"] = ":"

#Разные стили линий, чтобы на ч/б всё различалось
mpl.rcParams["axes.prop_cycle"] = mpl.cycler(
    linestyle=["-", ":", "-.", "--"],
    linewidth=[1.8, 1.8, 1.8, 1.8]
)

%matplotlib inline

In [2]:
#@title Установка TA-Lib в Colab
!wget http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
!tar -xzf ta-lib-0.4.0-src.tar.gz
%cd ta-lib
!./configure --prefix=/usr
!make
!make install
%cd ..
!pip install TA-Lib

#Импорт
import talib

--2026-04-26 05:48:30--  http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
Resolving prdownloads.sourceforge.net (prdownloads.sourceforge.net)... 104.18.12.149, 104.18.13.149, 2606:4700::6812:d95, ...
Connecting to prdownloads.sourceforge.net (prdownloads.sourceforge.net)|104.18.12.149|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: http://downloads.sourceforge.net/project/ta-lib/ta-lib/0.4.0/ta-lib-0.4.0-src.tar.gz [following]
--2026-04-26 05:48:31--  http://downloads.sourceforge.net/project/ta-lib/ta-lib/0.4.0/ta-lib-0.4.0-src.tar.gz
Resolving downloads.sourceforge.net (downloads.sourceforge.net)... 104.18.12.149, 104.18.13.149, 2606:4700::6812:c95, ...
Reusing existing connection to prdownloads.sourceforge.net:80.
HTTP request sent, awaiting response... 302 Found
Location: http://psychz.dl.sourceforge.net/project/ta-lib/ta-lib/0.4.0/ta-lib-0.4.0-src.tar.gz?viasf=1 [following]
--2026-04-26 05:48:31--  http://psychz.dl.sourcefo

In [3]:
#@title Mount Google Drive (для сохранения файлов)
drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive/Fuzzy+Indicators_tests"  # можно переименовать под проект
print("Артефакты будут сохраняться в:", BASE_DIR)

Mounted at /content/drive
Артефакты будут сохраняться в: /content/drive/MyDrive/Fuzzy+Indicators_tests


In [4]:
#@title Каталоги
os.makedirs(BASE_DIR, exist_ok=True)
for sub in ["data/raw", "data/processed", "preds", "metrics", "logs", "plots", "cfg"]:
    os.makedirs(f"{BASE_DIR}/{sub}", exist_ok=True)

# --- реестр (единая схема) ---
REGISTRY_CSV = f"{BASE_DIR}/metrics/registry.csv"
REGISTRY_COLS = [
    "model","ticker","step","lookback","units","window","seed",
    "rmse_train","mae_train","mape_train",
    "rmse_test","mae_test","mape_test",
    "train_time_s","infer_time_train_s","infer_time_test_s",
    "ts"
]
if not os.path.exists(REGISTRY_CSV):
    pd.DataFrame(columns=REGISTRY_COLS).to_csv(REGISTRY_CSV, index=False)

In [5]:
#@title Загрузка данных
os.makedirs(f"{BASE_DIR}/data/raw", exist_ok=True)

def _period_for(step: str) -> str:
    if step == "1m": return "7d"     #лимит Yahoo для 1m
    if step == "1h": return "730d"   #~2 года
    if step == "1d": return "max"
    raise ValueError("step ∈ {'1m','1h','1d'}")

def _cache_path_raw(ticker, step):
    t = str(ticker).replace('=','').replace('^','').replace('/','_')
    return f"{BASE_DIR}/data/raw/{t}_{step}.parquet"

def _load_yahoo_stable(name: str, step: str, retries: int = 3, delay: float = 1.0) -> pd.DataFrame:
    """
    Надёжная загрузка с yfinance.download + стандартизация.
    Возвращает df с колонками ['timestamp','Close'] в возрастающем порядке.
    """
    period = _period_for(step)
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(
                tickers=name, interval=step, period=period,
                auto_adjust=False, progress=False, threads=False
            )
            if df is None or df.empty:
                raise ValueError("Empty dataframe from yfinance")

            df = df.reset_index()
            time_col = None
            for c in ("Datetime", "Date", "index"):
                if c in df.columns:
                    time_col = c; break
            if time_col is None:
                if isinstance(df.index, pd.DatetimeIndex):
                    df = df.rename_axis("timestamp").reset_index()
                    time_col = "timestamp"
                else:
                    raise KeyError("No time column after reset_index")

            df = df.rename(columns={time_col: "timestamp"})
            if "Close" not in df.columns:
                #маленький фолбэк: иногда есть только Adj Close
                if "Adj Close" in df.columns:
                    df = df.rename(columns={"Adj Close":"Close"})
                else:
                    raise KeyError("No 'Close' column in downloaded frame")

            df = df[["timestamp", "Close"]].copy()
            df = df.sort_values("timestamp").reset_index(drop=True)
            if pd.api.types.is_datetime64_any_dtype(df["timestamp"]):
                try:
                    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)
                except Exception:
                    pass
            #sanity
            if len(df) < 10:
                raise ValueError(f"Too few rows: {len(df)}")

            return df
        except Exception as e:
            last_err = e
            print(f"[yfinance] attempt {attempt}/{retries} failed for {name} {step}: {e}")
            time.sleep(delay)

    raise RuntimeError(f"Не удалось загрузить {name} ({step}): {last_err}")

def get_historical_close_data(name: str, step: str, force_refresh: bool = False) -> pd.DataFrame:
    """
    Возвращает df(['timestamp','Close']) по времени ↑ и кэширует в Parquet.
    Если файл есть и force_refresh=False — читаем из кэша.
    """
    p = _cache_path_raw(name, step)
    if os.path.exists(p) and not force_refresh:
        try:
            df = pd.read_parquet(p)
            #проверим структуру
            if set(df.columns) >= {"timestamp","Close"} and len(df) > 0:
                df["timestamp"] = pd.to_datetime(df["timestamp"])
                df = df.sort_values("timestamp").reset_index(drop=True)
                return df[["timestamp","Close"]]
            else:
                print(f"[cache] некорректный формат {p} → перекачиваем")
        except Exception as e:
            print(f"[cache] не смог прочитать {p}: {e} → перекачиваем")

    df = _load_yahoo_stable(name, step)
    #гарантируем типы перед записью
    df = df.astype({"timestamp":"datetime64[ns]", "Close":"float64"})
    df.to_parquet(p, index=False)
    return df

In [6]:
#@title Обработка данных
def normalization(data):
    scaler = MinMaxScaler(feature_range = (0,1))
    data_norm = scaler.fit_transform(data)

    return data_norm

def de_normalization(data, new_data):
    scaler = MinMaxScaler(feature_range = (0,1))
    scaler.fit_transform(data)
    unormalized = scaler.inverse_transform(new_data)

    return unormalized

def split_train_test(data):

    #split into train and test sets
    train_size = int(len(data) * 0.75)
    train, test = data[0:train_size,], data[train_size:len(data),]
    return train, test

def create_dataset(dataset, lookback):
    dataX, dataY = [], []
    for i in range(len(dataset)-lookback-1):
      a = dataset[i:(i+lookback), 0]
      dataX.append(a)
      dataY.append(dataset[i + lookback, 0])
    return np.array(dataX), np.array(dataY)

def create_dataset_multifeature_1(dataset, lookback):
    dataX, dataY = [], []
    for i in range(len(dataset)-lookback-1):
        a = dataset[i:(i+lookback), :]     #теперь берем все фичи
        dataX.append(a)
        dataY.append(dataset[i + lookback, 0])  #целевая переменная — всё так же первая (норм. цена)
    return np.array(dataX), np.array(dataY)

def create_dataset_multifeature(dataset, lookback):
    dataX, dataY = [], []
    for i in range(len(dataset) - lookback - 1):
        a = dataset[i:(i + lookback), :]
        dataX.append(a)
        dataY.append(dataset[i + lookback, :])  #теперь y — вектор из 2 значений
    return np.array(dataX), np.array(dataY)


In [7]:
#@title Метрики
def mape_percent(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100.0)

In [8]:
#@title Basic FuzzyLSTM
def FuzzyBasic(step, unit, companies, lookback, window):
    df = get_historical_close_data(companies, step, force_refresh=True)
    close = df['Close'].values.reshape(-1, 1).astype(np.float32)

    train_close, test_close = split_train_test(close)
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_close)
    test_norm  = scaler.transform(test_close)

    #---------- каузальная фаззификация ----------
    #σ_t = std(window) сдвинутая на 1; phi_t = exp(-(Δx)^2 / (2 σ_{t-1}^2))
    def rolling_std_shift1(x, w):
        s = pd.Series(x.flatten())
        sig = s.rolling(w, min_periods=1).std().shift(1).bfill().values.astype(np.float32)
        sig[sig == 0] = 1e-8
        return sig

    # train
    sig_tr = rolling_std_shift1(train_norm, window)
    dx_tr = np.zeros_like(train_norm.flatten(), dtype=np.float32)
    if len(train_norm) > 1:
        dx_tr[1:] = train_norm.flatten()[1:] - train_norm.flatten()[:-1]
    phi_tr = np.exp(-(dx_tr**2) / (2.0 * (sig_tr**2) + 1e-8)).reshape(-1, 1).astype(np.float32)

    #test с прогревом окна хвостом из train
    tail = train_norm[-max(window-1, 1):] if len(train_norm) else np.empty((0,1), dtype=np.float32)
    test_in = np.vstack([tail, test_norm]) if len(tail) else test_norm
    sig_te_full = rolling_std_shift1(test_in, window)
    dx_te_full = np.zeros_like(test_in.flatten(), dtype=np.float32)
    if len(test_in) > 1:
        dx_te_full[1:] = test_in.flatten()[1:] - test_in.flatten()[:-1]
    phi_te_full = np.exp(-(dx_te_full**2) / (2.0 * (sig_te_full**2) + 1e-8)).reshape(-1, 1).astype(np.float32)
    phi_te = phi_te_full[len(tail):] if len(tail) else phi_te_full

    #---------- формируем 2-канальные ряды ----------
    train_dataset_fuzzy = np.hstack([train_norm, phi_tr]).astype(np.float32)  # [центр, phi]
    test_dataset_fuzzy  = np.hstack([test_norm,  phi_te]).astype(np.float32)

    #---------- окна X,y (цель — только цена) ----------
    train_X, train_y = create_dataset_multifeature_1(train_dataset_fuzzy, lookback)  # X: (L,2), y: норм. цена
    test_X,  test_y  = create_dataset_multifeature_1(test_dataset_fuzzy,  lookback)

    train_X = train_X.reshape(train_X.shape[0], train_X.shape[1], 2)
    test_X  = test_X.reshape(test_X.shape[0],  test_X.shape[1],  2)

    #---------- модель: параметрическая LSTM + дефаззификация y = μ*c_prev + (1-μ)*x̂ ----------
    inp = Input(shape=(lookback,2), name='inp')
    centers_seq = Lambda(lambda x: x[:, :, 0], name='centers_seq')(inp)   #(B,L)
    c_prev = Lambda(lambda c: c[:, -1:], name='c_prev')(centers_seq)      #последний центр окна (x_{i-1})

    h = RNN(ParametricLSTMCell_sigmoid(unit), name='fuzzy_inference')(inp)
    x_hat = Dense(1, name='x_hat')(h)                                    #прогноз (норм.)
    mu_hat = Dense(1, activation='sigmoid', name='mu_prev')(h)           #принадлежность к прошлому куполу ∈(0,1)
    y_out = Lambda(lambda t: t[1]*(1.0 - t[0]) + t[0]*t[2],
                   name='defuzz')([mu_hat, x_hat, c_prev])

    model = Model(inp, y_out, name='FuzzyLSTM_prev_defuzz')
    model.compile(optimizer='adam', loss='mse')

    t0 = time.time()
    hist = model.fit(train_X, train_y, epochs=100, batch_size=32,
                     validation_data=(test_X, test_y), verbose=False, shuffle=False)
    train_time_s = time.time() - t0

    #---------- предсказания + время ----------
    t1 = time.time()
    yhat_train = model.predict(train_X, verbose=False)
    infer_time_train_s = time.time() - t1

    t2 = time.time()
    yhat_test  = model.predict(test_X,  verbose=False)
    infer_time_test_s = time.time() - t2


    #денорм по цене
    y_train_true = scaler.inverse_transform(train_y.reshape(-1,1))
    y_test_true  = scaler.inverse_transform(test_y.reshape(-1,1))
    y_train_pred = scaler.inverse_transform(yhat_train)
    y_test_pred  = scaler.inverse_transform(yhat_test)

    #метрики
    rmse_train = math.sqrt(mean_squared_error(y_train_true, y_train_pred))
    mae_train  = mean_absolute_error(y_train_true, y_train_pred)
    mape_train = mape_train = mape_percent(y_train_true, y_train_pred)   # уже в %

    rmse_test = math.sqrt(mean_squared_error(y_test_true, y_test_pred))
    mae_test  = mean_absolute_error(y_test_true, y_test_pred)
    mape_test  = mape_percent(y_test_true,  y_test_pred)    # уже в %

    #графики
    fig1, ax1 = plt.subplots(figsize=(12,6))
    ax1.plot(df["timestamp"], close, label="data", alpha=0.6)
    ax1.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax1.legend(); ax1.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig1, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.png")

    fig2, ax2 = plt.subplots()
    ax2.plot(hist.history["loss"], label="train")
    ax2.plot(hist.history["val_loss"], label="val")
    ax2.legend(); ax2.set_title("Loss")
    save_plot(fig2, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_loss.png")

    #графики (test)
    fig3, ax3 = plt.subplots(figsize=(12,6))
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], close[-len(y_test_pred):], label="data", alpha=0.6)
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax3.legend(); ax3.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig3, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_test.png")

    #прогнозы CSV
    preds_df = pd.DataFrame({
        "timestamp": df["timestamp"].iloc[-len(y_test_pred):],
        "y_true": y_test_true.flatten(),
        "y_pred": y_test_pred.flatten()
    })
    save_preds(preds_df, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.csv")

    #в реестр
    registry_append({
        "model":"FuzzyBasic","ticker":companies,"step":step,
        "lookback":lookback,"units":unit,"window":window,"seed":42,
        "rmse_train":rmse_train,"mae_train":mae_train,"mape_train":mape_train,
        "rmse_test":rmse_test,"mae_test":mae_test,"mape_test":mape_test,
        "train_time_s":train_time_s,
        "infer_time_train_s":infer_time_train_s,
        "infer_time_test_s":infer_time_test_s,
        "ts": time.time()
    })

    print(f"TRAIN: RMSE={rmse_train:.4f} MAE={mae_train:.4f} MAPE={mape_train:.2f}%  | "
          f"time={train_time_s:.2f}s, infer={infer_time_train_s:.2f}s")
    print(f"TEST : RMSE={rmse_test:.4f} MAE={mae_test:.4f} MAPE={mape_test:.2f}%   | "
          f"infer={infer_time_test_s:.2f}s")

    return {
        "rmse_train": rmse_train, "mae_train": mae_train, "mape_train": mape_train,
        "rmse_test": rmse_test,   "mae_test": mae_test,   "mape_test": mape_test,
        "train_time_s": train_time_s,
        "infer_time_train_s": infer_time_train_s,
        "infer_time_test_s": infer_time_test_s
    }


class ParametricLSTMCell_sigmoid(LSTMCell):
    def build(self, input_shape):
        super().build(input_shape)
        #Добавляем параметры активации (α и β для сигмоиды)
        self.alpha = self.add_weight(
            name='alpha',
            shape=(1,),
            initializer=Constant(1.0),
            trainable=True
        )
        self.beta = self.add_weight(
            name='beta',
            shape=(1,),
            initializer=Constant(0.0),
            trainable=True
        )

    def param_sigmoid(self, x):
        return 1.0 / (1.0 + K.exp(-self.alpha * (x - self.beta)))

    def call(self, inputs, states, training=None):
        h_tm1 = states[0]  #предыдущий скрытый
        c_tm1 = states[1]  #предыдущий cell state

        z = K.dot(inputs, self.kernel)
        z += K.dot(h_tm1, self.recurrent_kernel)
        if self.use_bias:
            z = K.bias_add(z, self.bias)

        z0, z1, z2, z3 = tf.split(z, num_or_size_splits=4, axis=1)

        #Используем нашу сигмоиду
        i = self.param_sigmoid(z0)     #input gate
        f = self.param_sigmoid(z1)     #forget gate
        c = f * c_tm1 + i * K.tanh(z2) #cell state
        o = self.param_sigmoid(z3)     #output gate

        h = o * K.tanh(c)
        return h, [h, c]

In [9]:
#@title FuzzyLSTM + Momentum Indicators
def FuzzyMomentum(step, unit, companies, lookback, window):
    df = get_historical_close_data(companies, step, force_refresh=True)
    close = df['Close'].values.reshape(-1, 1).astype(np.float32)

    close_flat = close.astype(np.float64).flatten()

    #Параметры индикаторов
    rsi_period = 14

    trix_period = 30

    stochrsi_period = 14
    stochrsi_fastk_period = 5
    stochrsi_fastd_period = 3
    stochrsi_fastd_matype = 0

    #RSI
    rsi = talib.RSI(close_flat, timeperiod=rsi_period)

    #TRIX
    trix = talib.TRIX(close_flat, timeperiod=trix_period)

    #Stochastic RSI
    fastk, fastd = talib.STOCHRSI(close_flat, timeperiod=stochrsi_period, fastk_period=stochrsi_fastk_period, fastd_period=stochrsi_fastd_period, fastd_matype=stochrsi_fastd_matype)


    #Обрезаем начало, где индикаторы = NaN
    valid_mask = ~(np.isnan(rsi) | np.isnan(trix) | np.isnan(fastk))
    close = close[valid_mask]
    rsi = rsi[valid_mask].reshape(-1, 1).astype(np.float32)
    trix = trix[valid_mask].reshape(-1, 1).astype(np.float32)
    fastk = fastk[valid_mask].reshape(-1, 1).astype(np.float32)


    #Обновляем timestamp для соответствия
    df = df.iloc[valid_mask].reset_index(drop=True)

    train_close, test_close = split_train_test(close)
    train_rsi, test_rsi = split_train_test(rsi)
    train_trix, test_trix = split_train_test(trix)
    train_fastk, test_fastk = split_train_test(fastk)


    #---------- нормализация ----------
    scaler_close = MinMaxScaler(feature_range=(0, 1))
    scaler_rsi = MinMaxScaler(feature_range=(0, 1))
    scaler_trix = MinMaxScaler(feature_range=(0, 1))
    scaler_fastk = MinMaxScaler(feature_range=(0, 1))

    train_norm_close = scaler_close.fit_transform(train_close)
    test_norm_close  = scaler_close.transform(test_close)

    train_norm_rsi = scaler_rsi.fit_transform(train_rsi)
    test_norm_rsi  = scaler_rsi.transform(test_rsi)

    train_norm_trix = scaler_trix.fit_transform(train_trix)
    test_norm_trix  = scaler_trix.transform(test_trix)

    train_norm_fastk = scaler_fastk.fit_transform(train_fastk)
    test_norm_fastk  = scaler_fastk.transform(test_fastk)

    train_close, test_close = split_train_test(close)
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_close)
    test_norm  = scaler.transform(test_close)

    #---------- каузальная фаззификация ----------
    #σ_t = std(window) сдвинутая на 1; phi_t = exp(-(Δx)^2 / (2 σ_{t-1}^2))
    def rolling_std_shift1(x, w):
        s = pd.Series(x.flatten())
        sig = s.rolling(w, min_periods=1).std().shift(1).bfill().values.astype(np.float32)
        sig[sig == 0] = 1e-8
        return sig

    #train
    sig_tr = rolling_std_shift1(train_norm_close, window)
    dx_tr = np.zeros_like(train_norm_close.flatten(), dtype=np.float32)
    if len(train_norm_close) > 1:
        dx_tr[1:] = train_norm_close.flatten()[1:] - train_norm_close.flatten()[:-1]
    phi_tr = np.exp(-(dx_tr**2) / (2.0 * (sig_tr**2) + 1e-8)).reshape(-1, 1).astype(np.float32)

    #test с прогревом окна хвостом из train
    tail = train_norm_close[-max(window-1, 1):] if len(train_norm_close) else np.empty((0,1), dtype=np.float32)
    test_in = np.vstack([tail, test_norm_close]) if len(tail) else test_norm_close
    sig_te_full = rolling_std_shift1(test_in, window)
    dx_te_full = np.zeros_like(test_in.flatten(), dtype=np.float32)
    if len(test_in) > 1:
        dx_te_full[1:] = test_in.flatten()[1:] - test_in.flatten()[:-1]
    phi_te_full = np.exp(-(dx_te_full**2) / (2.0 * (sig_te_full**2) + 1e-8)).reshape(-1, 1).astype(np.float32)
    phi_te = phi_te_full[len(tail):] if len(tail) else phi_te_full

    #---------- формируем 5-канальные ряды ----------
    train_dataset = np.hstack([
        train_norm_close, phi_tr,
        train_norm_rsi, train_norm_trix, train_norm_fastk
    ]).astype(np.float32)

    test_dataset = np.hstack([
        test_norm_close, phi_te,
        test_norm_rsi, test_norm_trix, test_norm_fastk
    ]).astype(np.float32)

    #---------- окна X,y (цель — только цена) ----------
    train_X, train_y = create_dataset_multifeature_1(train_dataset, lookback)  # X: (L,5), y: норм. цена
    test_X,  test_y  = create_dataset_multifeature_1(test_dataset,  lookback)

    train_X = train_X.reshape(train_X.shape[0], train_X.shape[1], 5)  # 5 каналов
    test_X  = test_X.reshape(test_X.shape[0],  test_X.shape[1],  5)

    #---------- модель: параметрическая LSTM + дефаззификация y = μ*c_prev + (1-μ)*x̂ ----------
    inp = Input(shape=(lookback, 5), name='inp')
    centers_seq = Lambda(lambda x: x[:, :, 0], name='centers_seq')(inp)   #(B,L) - канал 0 = цена
    c_prev = Lambda(lambda c: c[:, -1:], name='c_prev')(centers_seq)      #последний центр окна (x_{i-1})

    h = RNN(ParametricLSTMCell_sigmoid(unit), name='fuzzy_inference')(inp)
    x_hat = Dense(1, name='x_hat')(h)                                    #прогноз (норм.)
    mu_hat = Dense(1, activation='sigmoid', name='mu_prev')(h)           #принадлежность к прошлому куполу ∈(0,1)
    y_out = Lambda(lambda t: t[1]*(1.0 - t[0]) + t[0]*t[2],
                   name='defuzz')([mu_hat, x_hat, c_prev])

    model = Model(inp, y_out, name='FuzzyLSTM_prev_defuzz')
    model.compile(optimizer='adam', loss='mse')

    t0 = time.time()
    hist = model.fit(train_X, train_y, epochs=100, batch_size=32,
                     validation_data=(test_X, test_y), verbose=False, shuffle=False)
    train_time_s = time.time() - t0

    #---------- предсказания + время ----------
    t1 = time.time()
    yhat_train = model.predict(train_X, verbose=False)
    infer_time_train_s = time.time() - t1

    t2 = time.time()
    yhat_test  = model.predict(test_X,  verbose=False)
    infer_time_test_s = time.time() - t2


    #денорм по цене
    y_train_true = scaler_close.inverse_transform(train_y.reshape(-1,1))
    y_test_true  = scaler_close.inverse_transform(test_y.reshape(-1,1))
    y_train_pred = scaler_close.inverse_transform(yhat_train)
    y_test_pred  = scaler_close.inverse_transform(yhat_test)


    #метрики
    rmse_train = math.sqrt(mean_squared_error(y_train_true, y_train_pred))
    mae_train  = mean_absolute_error(y_train_true, y_train_pred)
    mape_train = mape_train = mape_percent(y_train_true, y_train_pred)   #уже в %

    rmse_test = math.sqrt(mean_squared_error(y_test_true, y_test_pred))
    mae_test  = mean_absolute_error(y_test_true, y_test_pred)
    mape_test  = mape_percent(y_test_true,  y_test_pred)    #уже в %

    #графики
    fig1, ax1 = plt.subplots(figsize=(12,6))
    ax1.plot(df["timestamp"], close, label="data", alpha=0.6)
    ax1.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax1.legend(); ax1.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig1, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.png")

    fig2, ax2 = plt.subplots()
    ax2.plot(hist.history["loss"], label="train")
    ax2.plot(hist.history["val_loss"], label="val")
    ax2.legend(); ax2.set_title("Loss")
    save_plot(fig2, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_loss.png")

    #графики (test)
    fig3, ax3 = plt.subplots(figsize=(12,6))
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], close[-len(y_test_pred):], label="data", alpha=0.6)
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax3.legend(); ax3.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig3, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_test.png")

    #прогнозы CSV
    preds_df = pd.DataFrame({
        "timestamp": df["timestamp"].iloc[-len(y_test_pred):],
        "y_true": y_test_true.flatten(),
        "y_pred": y_test_pred.flatten()
    })
    save_preds(preds_df, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.csv")

    #в реестр
    registry_append({
        "model":"FuzzyMomentum","ticker":companies,"step":step,
        "lookback":lookback,"units":unit,"window":window,"seed":42,
        "rmse_train":rmse_train,"mae_train":mae_train,"mape_train":mape_train,
        "rmse_test":rmse_test,"mae_test":mae_test,"mape_test":mape_test,
        "train_time_s":train_time_s,
        "infer_time_train_s":infer_time_train_s,
        "infer_time_test_s":infer_time_test_s,
        "ts": time.time()
    })

    print(f"TRAIN: RMSE={rmse_train:.4f} MAE={mae_train:.4f} MAPE={mape_train:.2f}%  | "
          f"time={train_time_s:.2f}s, infer={infer_time_train_s:.2f}s")
    print(f"TEST : RMSE={rmse_test:.4f} MAE={mae_test:.4f} MAPE={mape_test:.2f}%   | "
          f"infer={infer_time_test_s:.2f}s")

    return {
        "rmse_train": rmse_train, "mae_train": mae_train, "mape_train": mape_train,
        "rmse_test": rmse_test,   "mae_test": mae_test,   "mape_test": mape_test,
        "train_time_s": train_time_s,
        "infer_time_train_s": infer_time_train_s,
        "infer_time_test_s": infer_time_test_s
    }


class ParametricLSTMCell_sigmoid(LSTMCell):
    def build(self, input_shape):
        super().build(input_shape)
        #Добавляем параметры активации (α и β для сигмоиды)
        self.alpha = self.add_weight(
            name='alpha',
            shape=(1,),
            initializer=Constant(1.0),
            trainable=True
        )
        self.beta = self.add_weight(
            name='beta',
            shape=(1,),
            initializer=Constant(0.0),
            trainable=True
        )

    def param_sigmoid(self, x):
        return 1.0 / (1.0 + K.exp(-self.alpha * (x - self.beta)))

    def call(self, inputs, states, training=None):
        h_tm1 = states[0]  #предыдущий скрытый
        c_tm1 = states[1]  #предыдущий cell state

        z = K.dot(inputs, self.kernel)
        z += K.dot(h_tm1, self.recurrent_kernel)
        if self.use_bias:
            z = K.bias_add(z, self.bias)

        z0, z1, z2, z3 = tf.split(z, num_or_size_splits=4, axis=1)

        #Используем нашу сигмоиду
        i = self.param_sigmoid(z0)     #input gate
        f = self.param_sigmoid(z1)     #forget gate
        c = f * c_tm1 + i * K.tanh(z2) #cell state
        o = self.param_sigmoid(z3)     #output gate

        h = o * K.tanh(c)
        return h, [h, c]

In [10]:
#@title FuzzyLSTM + Trend Indicators
def FuzzyTrend(step, unit, companies, lookback, window):

    df = get_historical_close_data(companies, step, force_refresh=True)
    close = df['Close'].values.reshape(-1, 1).astype(np.float32)

    #---------- расчёт индикаторов ----------
    close_flat = close.astype(np.float64).flatten()

    #Параметры индикаторов
    sma_period = 20

    ema_period = 20

    wma_period = 20

    macd_fastperiod = 12
    macd_slowperiod = 26
    macd_signalperiod = 9

    #Simple Moving Average
    sma = talib.SMA(close_flat, timeperiod=sma_period)

    #Exponential Moving Average
    ema = talib.EMA(close_flat, timeperiod=ema_period)

    #Weighted Moving Average
    wma = talib.WMA(close_flat, timeperiod=wma_period)

    #MACD
    macd, macd_signal, macd_hist = talib.MACD(close_flat, fastperiod=macd_fastperiod, slowperiod=macd_slowperiod, signalperiod=macd_signalperiod)

    #Обрезаем начало, где индикаторы = NaN
    valid_mask = ~(np.isnan(sma) | np.isnan(ema) | np.isnan(wma) |
                   np.isnan(macd) | np.isnan(macd_signal) | np.isnan(macd_hist))
    close = close[valid_mask]
    sma = sma[valid_mask].reshape(-1, 1).astype(np.float32)
    ema = ema[valid_mask].reshape(-1, 1).astype(np.float32)
    wma = wma[valid_mask].reshape(-1, 1).astype(np.float32)
    macd = macd[valid_mask].reshape(-1, 1).astype(np.float32)
    macd_signal = macd_signal[valid_mask].reshape(-1, 1).astype(np.float32)
    macd_hist = macd_hist[valid_mask].reshape(-1, 1).astype(np.float32)

    #Обновляем timestamp для соответствия
    df = df.iloc[valid_mask].reset_index(drop=True)

    train_close, test_close = split_train_test(close)
    train_sma, test_sma = split_train_test(sma)
    train_ema, test_ema = split_train_test(ema)
    train_wma, test_wma = split_train_test(wma)
    train_macd, test_macd = split_train_test(macd)
    train_macd_signal, test_macd_signal = split_train_test(macd_signal)
    train_macd_hist, test_macd_hist = split_train_test(macd_hist)

    #---------- нормализация ----------
    scaler_close = MinMaxScaler(feature_range=(0, 1))
    scaler_sma = MinMaxScaler(feature_range=(0, 1))
    scaler_ema = MinMaxScaler(feature_range=(0, 1))
    scaler_wma = MinMaxScaler(feature_range=(0, 1))
    scaler_macd = MinMaxScaler(feature_range=(0, 1))
    scaler_macd_signal = MinMaxScaler(feature_range=(0, 1))
    scaler_macd_hist = MinMaxScaler(feature_range=(0, 1))

    train_norm_close = scaler_close.fit_transform(train_close)
    test_norm_close  = scaler_close.transform(test_close)

    train_norm_sma = scaler_sma.fit_transform(train_sma)
    test_norm_sma  = scaler_sma.transform(test_sma)

    train_norm_ema = scaler_ema.fit_transform(train_ema)
    test_norm_ema  = scaler_ema.transform(test_ema)

    train_norm_wma = scaler_wma.fit_transform(train_wma)
    test_norm_wma  = scaler_wma.transform(test_wma)

    train_norm_macd = scaler_macd.fit_transform(train_macd)
    test_norm_macd  = scaler_macd.transform(test_macd)

    train_norm_macd_signal = scaler_macd_signal.fit_transform(train_macd_signal)
    test_norm_macd_signal  = scaler_macd_signal.transform(test_macd_signal)

    train_norm_macd_hist = scaler_macd_hist.fit_transform(train_macd_hist)
    test_norm_macd_hist  = scaler_macd_hist.transform(test_macd_hist)

    #---------- каузальная фаззификация ----------
    def rolling_std_shift1(x, w):
        s = pd.Series(x.flatten())
        sig = s.rolling(w, min_periods=1).std().shift(1).bfill().values.astype(np.float32)
        sig[sig == 0] = 1e-8
        return sig

    #train
    sig_tr = rolling_std_shift1(train_norm_close, window)
    dx_tr = np.zeros_like(train_norm_close.flatten(), dtype=np.float32)
    if len(train_norm_close) > 1:
        dx_tr[1:] = train_norm_close.flatten()[1:] - train_norm_close.flatten()[:-1]
    phi_tr = np.exp(-(dx_tr**2) / (2.0 * (sig_tr**2) + 1e-8)).reshape(-1, 1).astype(np.float32)

    #test с прогревом окна хвостом из train
    tail = train_norm_close[-max(window-1, 1):] if len(train_norm_close) else np.empty((0,1), dtype=np.float32)
    test_in = np.vstack([tail, test_norm_close]) if len(tail) else test_norm_close
    sig_te_full = rolling_std_shift1(test_in, window)
    dx_te_full = np.zeros_like(test_in.flatten(), dtype=np.float32)
    if len(test_in) > 1:
        dx_te_full[1:] = test_in.flatten()[1:] - test_in.flatten()[:-1]
    phi_te_full = np.exp(-(dx_te_full**2) / (2.0 * (sig_te_full**2) + 1e-8)).reshape(-1, 1).astype(np.float32)
    phi_te = phi_te_full[len(tail):] if len(tail) else phi_te_full

    #---------- формируем многоканальные ряды (8 каналов) ----------
    train_dataset = np.hstack([
        train_norm_close, phi_tr,
        train_norm_sma, train_norm_ema, train_norm_wma,
        train_norm_macd, train_norm_macd_signal, train_norm_macd_hist
    ]).astype(np.float32)

    test_dataset = np.hstack([
        test_norm_close, phi_te,
        test_norm_sma, test_norm_ema, test_norm_wma,
        test_norm_macd, test_norm_macd_signal, test_norm_macd_hist
    ]).astype(np.float32)

    #---------- окна X,y (цель — только цена) ----------
    train_X, train_y = create_dataset_multifeature_1(train_dataset, lookback)  # X: (L,8), y: норм. цена
    test_X,  test_y  = create_dataset_multifeature_1(test_dataset,  lookback)

    train_X = train_X.reshape(train_X.shape[0], train_X.shape[1], 8)  # 8 каналов
    test_X  = test_X.reshape(test_X.shape[0],  test_X.shape[1],  8)

    #---------- модель: параметрическая LSTM + дефаззификация y = μ*c_prev + (1-μ)*x̂ ----------
    inp = Input(shape=(lookback, 8), name='inp')
    centers_seq = Lambda(lambda x: x[:, :, 0], name='centers_seq')(inp)   #(B,L) - канал 0 = цена
    c_prev = Lambda(lambda c: c[:, -1:], name='c_prev')(centers_seq)      #последний центр окна (x_{i-1})

    h = RNN(ParametricLSTMCell_sigmoid(unit), name='fuzzy_inference')(inp)
    x_hat = Dense(1, name='x_hat')(h)                                    #прогноз (норм.)
    mu_hat = Dense(1, activation='sigmoid', name='mu_prev')(h)           #принадлежность к прошлому куполу ∈(0,1)
    y_out = Lambda(lambda t: t[1]*(1.0 - t[0]) + t[0]*t[2],
                   name='defuzz')([mu_hat, x_hat, c_prev])

    model = Model(inp, y_out, name='FuzzyLSTM_prev_defuzz')
    model.compile(optimizer='adam', loss='mse')

    t0 = time.time()
    hist = model.fit(train_X, train_y, epochs=100, batch_size=32,
                     validation_data=(test_X, test_y), verbose=False, shuffle=False)
    train_time_s = time.time() - t0

    #---------- предсказания + время ----------
    t1 = time.time()
    yhat_train = model.predict(train_X, verbose=False)
    infer_time_train_s = time.time() - t1

    t2 = time.time()
    yhat_test  = model.predict(test_X,  verbose=False)
    infer_time_test_s = time.time() - t2

    #денорм по цене
    y_train_true = scaler_close.inverse_transform(train_y.reshape(-1,1))
    y_test_true  = scaler_close.inverse_transform(test_y.reshape(-1,1))
    y_train_pred = scaler_close.inverse_transform(yhat_train)
    y_test_pred  = scaler_close.inverse_transform(yhat_test)

    #метрики
    rmse_train = math.sqrt(mean_squared_error(y_train_true, y_train_pred))
    mae_train  = mean_absolute_error(y_train_true, y_train_pred)
    mape_train = mape_percent(y_train_true, y_train_pred)

    rmse_test = math.sqrt(mean_squared_error(y_test_true, y_test_pred))
    mae_test  = mean_absolute_error(y_test_true, y_test_pred)
    mape_test  = mape_percent(y_test_true,  y_test_pred)

    #графики
    fig1, ax1 = plt.subplots(figsize=(12,6))
    ax1.plot(df["timestamp"], close, label="data", alpha=0.6)
    ax1.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax1.legend(); ax1.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig1, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.png")

    fig2, ax2 = plt.subplots()
    ax2.plot(hist.history["loss"], label="train")
    ax2.plot(hist.history["val_loss"], label="val")
    ax2.legend(); ax2.set_title("Loss")
    save_plot(fig2, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_loss.png")

    fig3, ax3 = plt.subplots(figsize=(12,6))
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], close[-len(y_test_pred):], label="data", alpha=0.6)
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax3.legend(); ax3.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig3, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_test.png")

    #прогнозы CSV
    preds_df = pd.DataFrame({
        "timestamp": df["timestamp"].iloc[-len(y_test_pred):],
        "y_true": y_test_true.flatten(),
        "y_pred": y_test_pred.flatten()
    })
    save_preds(preds_df, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.csv")

    #в реестр
    registry_append({
        "model":"FuzzyTrend","ticker":companies,"step":step,
        "lookback":lookback,"units":unit,"window":window,"seed":42,
        "rmse_train":rmse_train,"mae_train":mae_train,"mape_train":mape_train,
        "rmse_test":rmse_test,"mae_test":mae_test,"mape_test":mape_test,
        "train_time_s":train_time_s,
        "infer_time_train_s":infer_time_train_s,
        "infer_time_test_s":infer_time_test_s,
        "ts": time.time()
    })

    print(f"TRAIN: RMSE={rmse_train:.4f} MAE={mae_train:.4f} MAPE={mape_train:.2f}%  | "
          f"time={train_time_s:.2f}s, infer={infer_time_train_s:.2f}s")
    print(f"TEST : RMSE={rmse_test:.4f} MAE={mae_test:.4f} MAPE={mape_test:.2f}%   | "
          f"infer={infer_time_test_s:.2f}s")

    return {
        "rmse_train": rmse_train, "mae_train": mae_train, "mape_train": mape_train,
        "rmse_test": rmse_test,   "mae_test": mae_test,   "mape_test": mape_test,
        "train_time_s": train_time_s,
        "infer_time_train_s": infer_time_train_s,
        "infer_time_test_s": infer_time_test_s
    }


class ParametricLSTMCell_sigmoid(LSTMCell):
    def build(self, input_shape):
        super().build(input_shape)
        #Добавляем параметры активации (α и β для сигмоиды)
        self.alpha = self.add_weight(
            name='alpha',
            shape=(1,),
            initializer=Constant(1.0),
            trainable=True
        )
        self.beta = self.add_weight(
            name='beta',
            shape=(1,),
            initializer=Constant(0.0),
            trainable=True
        )

    def param_sigmoid(self, x):
        return 1.0 / (1.0 + K.exp(-self.alpha * (x - self.beta)))

    def call(self, inputs, states, training=None):
        h_tm1 = states[0]  #предыдущий скрытый
        c_tm1 = states[1]  #предыдущий cell state

        z = K.dot(inputs, self.kernel)
        z += K.dot(h_tm1, self.recurrent_kernel)
        if self.use_bias:
            z = K.bias_add(z, self.bias)

        z0, z1, z2, z3 = tf.split(z, num_or_size_splits=4, axis=1)

        #Используем нашу сигмоиду
        i = self.param_sigmoid(z0)     #input gate
        f = self.param_sigmoid(z1)     #forget gate
        c = f * c_tm1 + i * K.tanh(z2) #cell state
        o = self.param_sigmoid(z3)     #output gate

        h = o * K.tanh(c)
        return h, [h, c]

In [11]:
#@title FuzzyLSTM + Volatility Indicators
def FuzzyVolatility(step, unit, companies, lookback, window):

    df = get_historical_close_data(companies, step, force_refresh=True)
    close = df['Close'].values.reshape(-1, 1).astype(np.float32)

    #---------- расчёт индикаторов ----------
    close_flat = close.astype(np.float64).flatten()

    #Параметры индикаторов
    stddev_period = 20
    stddev_nbdev = 1

    b_bands_period = 20
    b_bands_nbdevup = 2
    b_bands_nbdevdn = 2
    b_bands_matype = 0

    #Standard Deviation
    stddev = talib.STDDEV(close_flat, timeperiod=stddev_period, nbdev=stddev_nbdev)

    #Bollinger Bands
    bb_upper, bb_middle, bb_lower = talib.BBANDS(close_flat, timeperiod=b_bands_period, nbdevup=b_bands_nbdevup, nbdevdn=b_bands_nbdevdn, matype=b_bands_matype)

    #Bollinger %B = (price - lower) / (upper - lower)
    bb_percent_b = (close_flat - bb_lower) / (bb_upper - bb_lower + 1e-8)

    #Обрезаем начало, где индикаторы = NaN
    valid_mask = ~(np.isnan(stddev) | np.isnan(bb_upper) | np.isnan(bb_lower) | np.isnan(bb_percent_b))
    close = close[valid_mask]
    stddev = stddev[valid_mask].reshape(-1, 1).astype(np.float32)
    bb_upper = bb_upper[valid_mask].reshape(-1, 1).astype(np.float32)
    bb_lower = bb_lower[valid_mask].reshape(-1, 1).astype(np.float32)
    bb_percent_b = bb_percent_b[valid_mask].reshape(-1, 1).astype(np.float32)

    #Обновляем timestamp для соответствия
    df = df.iloc[valid_mask].reset_index(drop=True)

    train_close, test_close = split_train_test(close)
    train_stddev, test_stddev = split_train_test(stddev)
    train_bb_upper, test_bb_upper = split_train_test(bb_upper)
    train_bb_lower, test_bb_lower = split_train_test(bb_lower)
    train_bb_percent_b, test_bb_percent_b = split_train_test(bb_percent_b)

    #---------- нормализация ----------
    scaler_close = MinMaxScaler(feature_range=(0, 1))
    scaler_stddev = MinMaxScaler(feature_range=(0, 1))
    scaler_bb_upper = MinMaxScaler(feature_range=(0, 1))
    scaler_bb_lower = MinMaxScaler(feature_range=(0, 1))
    scaler_bb_percent_b = MinMaxScaler(feature_range=(0, 1))

    train_norm_close = scaler_close.fit_transform(train_close)
    test_norm_close  = scaler_close.transform(test_close)

    train_norm_stddev = scaler_stddev.fit_transform(train_stddev)
    test_norm_stddev  = scaler_stddev.transform(test_stddev)

    train_norm_bb_upper = scaler_bb_upper.fit_transform(train_bb_upper)
    test_norm_bb_upper  = scaler_bb_upper.transform(test_bb_upper)

    train_norm_bb_lower = scaler_bb_lower.fit_transform(train_bb_lower)
    test_norm_bb_lower  = scaler_bb_lower.transform(test_bb_lower)

    train_norm_bb_percent_b = scaler_bb_percent_b.fit_transform(train_bb_percent_b)
    test_norm_bb_percent_b  = scaler_bb_percent_b.transform(test_bb_percent_b)

    #---------- каузальная фаззификация ----------
    def rolling_std_shift1(x, w):
        s = pd.Series(x.flatten())
        sig = s.rolling(w, min_periods=1).std().shift(1).bfill().values.astype(np.float32)
        sig[sig == 0] = 1e-8
        return sig

    #train
    sig_tr = rolling_std_shift1(train_norm_close, window)
    dx_tr = np.zeros_like(train_norm_close.flatten(), dtype=np.float32)
    if len(train_norm_close) > 1:
        dx_tr[1:] = train_norm_close.flatten()[1:] - train_norm_close.flatten()[:-1]
    phi_tr = np.exp(-(dx_tr**2) / (2.0 * (sig_tr**2) + 1e-8)).reshape(-1, 1).astype(np.float32)

    #test с прогревом окна хвостом из train
    tail = train_norm_close[-max(window-1, 1):] if len(train_norm_close) else np.empty((0,1), dtype=np.float32)
    test_in = np.vstack([tail, test_norm_close]) if len(tail) else test_norm_close
    sig_te_full = rolling_std_shift1(test_in, window)
    dx_te_full = np.zeros_like(test_in.flatten(), dtype=np.float32)
    if len(test_in) > 1:
        dx_te_full[1:] = test_in.flatten()[1:] - test_in.flatten()[:-1]
    phi_te_full = np.exp(-(dx_te_full**2) / (2.0 * (sig_te_full**2) + 1e-8)).reshape(-1, 1).astype(np.float32)
    phi_te = phi_te_full[len(tail):] if len(tail) else phi_te_full

    #---------- формируем многоканальные ряды (6 каналов) ----------
    train_dataset = np.hstack([
        train_norm_close, phi_tr,
        train_norm_stddev, train_norm_bb_upper, train_norm_bb_lower, train_norm_bb_percent_b
    ]).astype(np.float32)

    test_dataset = np.hstack([
        test_norm_close, phi_te,
        test_norm_stddev, test_norm_bb_upper, test_norm_bb_lower, test_norm_bb_percent_b
    ]).astype(np.float32)

    #---------- окна X,y (цель — только цена) ----------
    train_X, train_y = create_dataset_multifeature_1(train_dataset, lookback)  # X: (L,6), y: норм. цена
    test_X,  test_y  = create_dataset_multifeature_1(test_dataset,  lookback)

    train_X = train_X.reshape(train_X.shape[0], train_X.shape[1], 6)  # 6 каналов
    test_X  = test_X.reshape(test_X.shape[0],  test_X.shape[1],  6)

    #---------- модель: параметрическая LSTM + дефаззификация y = μ*c_prev + (1-μ)*x̂ ----------
    inp = Input(shape=(lookback, 6), name='inp')
    centers_seq = Lambda(lambda x: x[:, :, 0], name='centers_seq')(inp)   #(B,L) - канал 0 = цена
    c_prev = Lambda(lambda c: c[:, -1:], name='c_prev')(centers_seq)      #последний центр окна (x_{i-1})

    h = RNN(ParametricLSTMCell_sigmoid(unit), name='fuzzy_inference')(inp)
    x_hat = Dense(1, name='x_hat')(h)                                    #прогноз (норм.)
    mu_hat = Dense(1, activation='sigmoid', name='mu_prev')(h)           #принадлежность к прошлому куполу ∈(0,1)
    y_out = Lambda(lambda t: t[1]*(1.0 - t[0]) + t[0]*t[2],
                   name='defuzz')([mu_hat, x_hat, c_prev])

    model = Model(inp, y_out, name='FuzzyLSTM_prev_defuzz')
    model.compile(optimizer='adam', loss='mse')

    t0 = time.time()
    hist = model.fit(train_X, train_y, epochs=100, batch_size=32,
                     validation_data=(test_X, test_y), verbose=False, shuffle=False)
    train_time_s = time.time() - t0

    #---------- предсказания + время ----------
    t1 = time.time()
    yhat_train = model.predict(train_X, verbose=False)
    infer_time_train_s = time.time() - t1

    t2 = time.time()
    yhat_test  = model.predict(test_X,  verbose=False)
    infer_time_test_s = time.time() - t2

    #денорм по цене
    y_train_true = scaler_close.inverse_transform(train_y.reshape(-1,1))
    y_test_true  = scaler_close.inverse_transform(test_y.reshape(-1,1))
    y_train_pred = scaler_close.inverse_transform(yhat_train)
    y_test_pred  = scaler_close.inverse_transform(yhat_test)

    #метрики
    rmse_train = math.sqrt(mean_squared_error(y_train_true, y_train_pred))
    mae_train  = mean_absolute_error(y_train_true, y_train_pred)
    mape_train = mape_percent(y_train_true, y_train_pred)

    rmse_test = math.sqrt(mean_squared_error(y_test_true, y_test_pred))
    mae_test  = mean_absolute_error(y_test_true, y_test_pred)
    mape_test  = mape_percent(y_test_true,  y_test_pred)

    #графики
    fig1, ax1 = plt.subplots(figsize=(12,6))
    ax1.plot(df["timestamp"], close, label="data", alpha=0.6)
    ax1.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax1.legend(); ax1.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig1, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.png")

    fig2, ax2 = plt.subplots()
    ax2.plot(hist.history["loss"], label="train")
    ax2.plot(hist.history["val_loss"], label="val")
    ax2.legend(); ax2.set_title("Loss")
    save_plot(fig2, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_loss.png")

    fig3, ax3 = plt.subplots(figsize=(12,6))
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], close[-len(y_test_pred):], label="data", alpha=0.6)
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax3.legend(); ax3.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig3, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_test.png")

    #прогнозы CSV
    preds_df = pd.DataFrame({
        "timestamp": df["timestamp"].iloc[-len(y_test_pred):],
        "y_true": y_test_true.flatten(),
        "y_pred": y_test_pred.flatten()
    })
    save_preds(preds_df, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.csv")

    #в реестр
    registry_append({
        "model":"FuzzyVolatility","ticker":companies,"step":step,
        "lookback":lookback,"units":unit,"window":window,"seed":42,
        "rmse_train":rmse_train,"mae_train":mae_train,"mape_train":mape_train,
        "rmse_test":rmse_test,"mae_test":mae_test,"mape_test":mape_test,
        "train_time_s":train_time_s,
        "infer_time_train_s":infer_time_train_s,
        "infer_time_test_s":infer_time_test_s,
        "ts": time.time()
    })

    print(f"TRAIN: RMSE={rmse_train:.4f} MAE={mae_train:.4f} MAPE={mape_train:.2f}%  | "
          f"time={train_time_s:.2f}s, infer={infer_time_train_s:.2f}s")
    print(f"TEST : RMSE={rmse_test:.4f} MAE={mae_test:.4f} MAPE={mape_test:.2f}%   | "
          f"infer={infer_time_test_s:.2f}s")

    return {
        "rmse_train": rmse_train, "mae_train": mae_train, "mape_train": mape_train,
        "rmse_test": rmse_test,   "mae_test": mae_test,   "mape_test": mape_test,
        "train_time_s": train_time_s,
        "infer_time_train_s": infer_time_train_s,
        "infer_time_test_s": infer_time_test_s
    }


class ParametricLSTMCell_sigmoid(LSTMCell):
    def build(self, input_shape):
        super().build(input_shape)
        #Добавляем параметры активации (α и β для сигмоиды)
        self.alpha = self.add_weight(
            name='alpha',
            shape=(1,),
            initializer=Constant(1.0),
            trainable=True
        )
        self.beta = self.add_weight(
            name='beta',
            shape=(1,),
            initializer=Constant(0.0),
            trainable=True
        )

    def param_sigmoid(self, x):
        return 1.0 / (1.0 + K.exp(-self.alpha * (x - self.beta)))

    def call(self, inputs, states, training=None):
        h_tm1 = states[0]  #предыдущий скрытый
        c_tm1 = states[1]  #предыдущий cell state

        z = K.dot(inputs, self.kernel)
        z += K.dot(h_tm1, self.recurrent_kernel)
        if self.use_bias:
            z = K.bias_add(z, self.bias)

        z0, z1, z2, z3 = tf.split(z, num_or_size_splits=4, axis=1)

        #Используем нашу сигмоиду
        i = self.param_sigmoid(z0)     #input gate
        f = self.param_sigmoid(z1)     #forget gate
        c = f * c_tm1 + i * K.tanh(z2) #cell state
        o = self.param_sigmoid(z3)     #output gate

        h = o * K.tanh(c)
        return h, [h, c]

In [12]:
#@title FuzzyLSTM + All Indicators
def FuzzyAll(step, unit, companies, lookback, window):

    df = get_historical_close_data(companies, step, force_refresh=True)
    close = df['Close'].values.reshape(-1, 1).astype(np.float32)

    #---------- расчёт всех индикаторов ----------
    close_flat = close.astype(np.float64).flatten()

    #Параметры индикаторов
    rsi_period = 14

    trix_period = 30

    stochrsi_period = 14
    stochrsi_fastk_period = 5
    stochrsi_fastd_period = 3
    stochrsi_fastd_matype = 0


    sma_period = 20

    ema_period = 20

    wma_period = 20

    macd_fastperiod = 12
    macd_slowperiod = 26
    macd_signalperiod = 9


    stddev_period = 20
    stddev_nbdev = 1

    b_bands_period = 20
    b_bands_nbdevup = 2
    b_bands_nbdevdn = 2
    b_bands_matype = 0


    #RSI
    rsi = talib.RSI(close_flat, timeperiod=rsi_period)

    #TRIX
    trix = talib.TRIX(close_flat, timeperiod=trix_period)

    #Stochastic RSI
    fastk, fastd = talib.STOCHRSI(close_flat, timeperiod=stochrsi_period, fastk_period=stochrsi_fastk_period, fastd_period=stochrsi_fastd_period, fastd_matype=stochrsi_fastd_matype)


    #Simple Moving Average
    sma = talib.SMA(close_flat, timeperiod=sma_period)

    #Exponential Moving Average
    ema = talib.EMA(close_flat, timeperiod=ema_period)

    #Weighted Moving Average
    wma = talib.WMA(close_flat, timeperiod=wma_period)

    #MACD
    macd, macd_signal, macd_hist = talib.MACD(close_flat, fastperiod=macd_fastperiod, slowperiod=macd_slowperiod, signalperiod=macd_signalperiod)


    #Standard Deviation
    stddev = talib.STDDEV(close_flat, timeperiod=stddev_period, nbdev=stddev_nbdev)

    #Bollinger Bands
    bb_upper, bb_middle, bb_lower = talib.BBANDS(close_flat, timeperiod=b_bands_period, nbdevup=b_bands_nbdevup, nbdevdn=b_bands_nbdevdn, matype=b_bands_matype)

    #Bollinger %B = (price - lower) / (upper - lower)
    bb_percent_b = (close_flat - bb_lower) / (bb_upper - bb_lower + 1e-8)

    #Обрезаем начало, где любой индикатор = NaN
    valid_mask = ~(np.isnan(rsi) | np.isnan(trix) | np.isnan(fastk) |
                   np.isnan(sma) | np.isnan(ema) | np.isnan(wma) |
                   np.isnan(macd) | np.isnan(macd_signal) | np.isnan(macd_hist) |
                   np.isnan(stddev) | np.isnan(bb_upper) | np.isnan(bb_lower) | np.isnan(bb_percent_b))

    close = close[valid_mask]
    rsi = rsi[valid_mask].reshape(-1, 1).astype(np.float32)
    trix = trix[valid_mask].reshape(-1, 1).astype(np.float32)
    fastk = fastk[valid_mask].reshape(-1, 1).astype(np.float32)
    sma = sma[valid_mask].reshape(-1, 1).astype(np.float32)
    ema = ema[valid_mask].reshape(-1, 1).astype(np.float32)
    wma = wma[valid_mask].reshape(-1, 1).astype(np.float32)
    macd = macd[valid_mask].reshape(-1, 1).astype(np.float32)
    macd_signal = macd_signal[valid_mask].reshape(-1, 1).astype(np.float32)
    macd_hist = macd_hist[valid_mask].reshape(-1, 1).astype(np.float32)
    stddev = stddev[valid_mask].reshape(-1, 1).astype(np.float32)
    bb_upper = bb_upper[valid_mask].reshape(-1, 1).astype(np.float32)
    bb_lower = bb_lower[valid_mask].reshape(-1, 1).astype(np.float32)
    bb_percent_b = bb_percent_b[valid_mask].reshape(-1, 1).astype(np.float32)

    #Обновляем timestamp для соответствия
    df = df.iloc[valid_mask].reset_index(drop=True)

    train_close, test_close = split_train_test(close)
    train_rsi, test_rsi = split_train_test(rsi)
    train_trix, test_trix = split_train_test(trix)
    train_fastk, test_fastk = split_train_test(fastk)
    train_sma, test_sma = split_train_test(sma)
    train_ema, test_ema = split_train_test(ema)
    train_wma, test_wma = split_train_test(wma)
    train_macd, test_macd = split_train_test(macd)
    train_macd_signal, test_macd_signal = split_train_test(macd_signal)
    train_macd_hist, test_macd_hist = split_train_test(macd_hist)
    train_stddev, test_stddev = split_train_test(stddev)
    train_bb_upper, test_bb_upper = split_train_test(bb_upper)
    train_bb_lower, test_bb_lower = split_train_test(bb_lower)
    train_bb_percent_b, test_bb_percent_b = split_train_test(bb_percent_b)

    #---------- нормализация ----------
    scaler_close = MinMaxScaler(feature_range=(0, 1))
    scaler_rsi = MinMaxScaler(feature_range=(0, 1))
    scaler_trix = MinMaxScaler(feature_range=(0, 1))
    scaler_fastk = MinMaxScaler(feature_range=(0, 1))
    scaler_sma = MinMaxScaler(feature_range=(0, 1))
    scaler_ema = MinMaxScaler(feature_range=(0, 1))
    scaler_wma = MinMaxScaler(feature_range=(0, 1))
    scaler_macd = MinMaxScaler(feature_range=(0, 1))
    scaler_macd_signal = MinMaxScaler(feature_range=(0, 1))
    scaler_macd_hist = MinMaxScaler(feature_range=(0, 1))
    scaler_stddev = MinMaxScaler(feature_range=(0, 1))
    scaler_bb_upper = MinMaxScaler(feature_range=(0, 1))
    scaler_bb_lower = MinMaxScaler(feature_range=(0, 1))
    scaler_bb_percent_b = MinMaxScaler(feature_range=(0, 1))

    train_norm_close = scaler_close.fit_transform(train_close)
    test_norm_close  = scaler_close.transform(test_close)

    train_norm_rsi = scaler_rsi.fit_transform(train_rsi)
    test_norm_rsi  = scaler_rsi.transform(test_rsi)

    train_norm_trix = scaler_trix.fit_transform(train_trix)
    test_norm_trix  = scaler_trix.transform(test_trix)

    train_norm_fastk = scaler_fastk.fit_transform(train_fastk)
    test_norm_fastk  = scaler_fastk.transform(test_fastk)

    train_norm_sma = scaler_sma.fit_transform(train_sma)
    test_norm_sma  = scaler_sma.transform(test_sma)

    train_norm_ema = scaler_ema.fit_transform(train_ema)
    test_norm_ema  = scaler_ema.transform(test_ema)

    train_norm_wma = scaler_wma.fit_transform(train_wma)
    test_norm_wma  = scaler_wma.transform(test_wma)

    train_norm_macd = scaler_macd.fit_transform(train_macd)
    test_norm_macd  = scaler_macd.transform(test_macd)

    train_norm_macd_signal = scaler_macd_signal.fit_transform(train_macd_signal)
    test_norm_macd_signal  = scaler_macd_signal.transform(test_macd_signal)

    train_norm_macd_hist = scaler_macd_hist.fit_transform(train_macd_hist)
    test_norm_macd_hist  = scaler_macd_hist.transform(test_macd_hist)

    train_norm_stddev = scaler_stddev.fit_transform(train_stddev)
    test_norm_stddev  = scaler_stddev.transform(test_stddev)

    train_norm_bb_upper = scaler_bb_upper.fit_transform(train_bb_upper)
    test_norm_bb_upper  = scaler_bb_upper.transform(test_bb_upper)

    train_norm_bb_lower = scaler_bb_lower.fit_transform(train_bb_lower)
    test_norm_bb_lower  = scaler_bb_lower.transform(test_bb_lower)

    train_norm_bb_percent_b = scaler_bb_percent_b.fit_transform(train_bb_percent_b)
    test_norm_bb_percent_b  = scaler_bb_percent_b.transform(test_bb_percent_b)

    #---------- каузальная фаззификация ----------
    def rolling_std_shift1(x, w):
        s = pd.Series(x.flatten())
        sig = s.rolling(w, min_periods=1).std().shift(1).bfill().values.astype(np.float32)
        sig[sig == 0] = 1e-8
        return sig

    #train
    sig_tr = rolling_std_shift1(train_norm_close, window)
    dx_tr = np.zeros_like(train_norm_close.flatten(), dtype=np.float32)
    if len(train_norm_close) > 1:
        dx_tr[1:] = train_norm_close.flatten()[1:] - train_norm_close.flatten()[:-1]
    phi_tr = np.exp(-(dx_tr**2) / (2.0 * (sig_tr**2) + 1e-8)).reshape(-1, 1).astype(np.float32)

    #test с прогревом окна хвостом из train
    tail = train_norm_close[-max(window-1, 1):] if len(train_norm_close) else np.empty((0,1), dtype=np.float32)
    test_in = np.vstack([tail, test_norm_close]) if len(tail) else test_norm_close
    sig_te_full = rolling_std_shift1(test_in, window)
    dx_te_full = np.zeros_like(test_in.flatten(), dtype=np.float32)
    if len(test_in) > 1:
        dx_te_full[1:] = test_in.flatten()[1:] - test_in.flatten()[:-1]
    phi_te_full = np.exp(-(dx_te_full**2) / (2.0 * (sig_te_full**2) + 1e-8)).reshape(-1, 1).astype(np.float32)
    phi_te = phi_te_full[len(tail):] if len(tail) else phi_te_full

    #---------- формируем многоканальные ряды (15 каналов) ----------
    train_dataset = np.hstack([
        train_norm_close, phi_tr,
        train_norm_rsi, train_norm_trix, train_norm_fastk,
        train_norm_sma, train_norm_ema, train_norm_wma,
        train_norm_macd, train_norm_macd_signal, train_norm_macd_hist,
        train_norm_stddev, train_norm_bb_upper, train_norm_bb_lower, train_norm_bb_percent_b
    ]).astype(np.float32)

    test_dataset = np.hstack([
        test_norm_close, phi_te,
        test_norm_rsi, test_norm_trix, test_norm_fastk,
        test_norm_sma, test_norm_ema, test_norm_wma,
        test_norm_macd, test_norm_macd_signal, test_norm_macd_hist,
        test_norm_stddev, test_norm_bb_upper, test_norm_bb_lower, test_norm_bb_percent_b
    ]).astype(np.float32)

    #---------- окна X,y (цель — только цена) ----------
    train_X, train_y = create_dataset_multifeature_1(train_dataset, lookback)  # X: (L,15), y: норм. цена
    test_X,  test_y  = create_dataset_multifeature_1(test_dataset,  lookback)

    train_X = train_X.reshape(train_X.shape[0], train_X.shape[1], 15)  # 15 каналов
    test_X  = test_X.reshape(test_X.shape[0],  test_X.shape[1],  15)

    #---------- модель: параметрическая LSTM + дефаззификация y = μ*c_prev + (1-μ)*x̂ ----------
    inp = Input(shape=(lookback, 15), name='inp')
    centers_seq = Lambda(lambda x: x[:, :, 0], name='centers_seq')(inp)   #(B,L) - канал 0 = цена
    c_prev = Lambda(lambda c: c[:, -1:], name='c_prev')(centers_seq)      #последний центр окна (x_{i-1})

    h = RNN(ParametricLSTMCell_sigmoid(unit), name='fuzzy_inference')(inp)
    x_hat = Dense(1, name='x_hat')(h)                                    #прогноз (норм.)
    mu_hat = Dense(1, activation='sigmoid', name='mu_prev')(h)           #принадлежность к прошлому куполу ∈(0,1)
    y_out = Lambda(lambda t: t[1]*(1.0 - t[0]) + t[0]*t[2],
                   name='defuzz')([mu_hat, x_hat, c_prev])

    model = Model(inp, y_out, name='FuzzyLSTM_prev_defuzz')
    model.compile(optimizer='adam', loss='mse')

    t0 = time.time()
    hist = model.fit(train_X, train_y, epochs=100, batch_size=32,
                     validation_data=(test_X, test_y), verbose=False, shuffle=False)
    train_time_s = time.time() - t0

    #---------- предсказания + время ----------
    t1 = time.time()
    yhat_train = model.predict(train_X, verbose=False)
    infer_time_train_s = time.time() - t1

    t2 = time.time()
    yhat_test  = model.predict(test_X,  verbose=False)
    infer_time_test_s = time.time() - t2

    #денорм по цене
    y_train_true = scaler_close.inverse_transform(train_y.reshape(-1,1))
    y_test_true  = scaler_close.inverse_transform(test_y.reshape(-1,1))
    y_train_pred = scaler_close.inverse_transform(yhat_train)
    y_test_pred  = scaler_close.inverse_transform(yhat_test)

    #метрики
    rmse_train = math.sqrt(mean_squared_error(y_train_true, y_train_pred))
    mae_train  = mean_absolute_error(y_train_true, y_train_pred)
    mape_train = mape_percent(y_train_true, y_train_pred)

    rmse_test = math.sqrt(mean_squared_error(y_test_true, y_test_pred))
    mae_test  = mean_absolute_error(y_test_true, y_test_pred)
    mape_test  = mape_percent(y_test_true,  y_test_pred)

    #графики
    fig1, ax1 = plt.subplots(figsize=(12,6))
    ax1.plot(df["timestamp"], close, label="data", alpha=0.6)
    ax1.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax1.legend(); ax1.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig1, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.png")

    fig2, ax2 = plt.subplots()
    ax2.plot(hist.history["loss"], label="train")
    ax2.plot(hist.history["val_loss"], label="val")
    ax2.legend(); ax2.set_title("Loss")
    save_plot(fig2, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_loss.png")

    fig3, ax3 = plt.subplots(figsize=(12,6))
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], close[-len(y_test_pred):], label="data", alpha=0.6)
    ax3.plot(df["timestamp"].iloc[-len(y_test_pred):], y_test_pred, label="test")
    ax3.legend(); ax3.set_title(f"{companies} {step} — Fuzzy LSTM (w={window}, lb={lookback}, u={unit})")
    save_plot(fig3, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}_test.png")

    #прогнозы CSV
    preds_df = pd.DataFrame({
        "timestamp": df["timestamp"].iloc[-len(y_test_pred):],
        "y_true": y_test_true.flatten(),
        "y_pred": y_test_pred.flatten()
    })
    save_preds(preds_df, f"FuzzyLSTM_{companies}_{step}_lb{lookback}_u{unit}_w{window}.csv")

    #в реестр
    registry_append({
        "model":"FuzzyAll","ticker":companies,"step":step,
        "lookback":lookback,"units":unit,"window":window,"seed":42,
        "rmse_train":rmse_train,"mae_train":mae_train,"mape_train":mape_train,
        "rmse_test":rmse_test,"mae_test":mae_test,"mape_test":mape_test,
        "train_time_s":train_time_s,
        "infer_time_train_s":infer_time_train_s,
        "infer_time_test_s":infer_time_test_s,
        "ts": time.time()
    })

    print(f"TRAIN: RMSE={rmse_train:.4f} MAE={mae_train:.4f} MAPE={mape_train:.2f}%  | "
          f"time={train_time_s:.2f}s, infer={infer_time_train_s:.2f}s")
    print(f"TEST : RMSE={rmse_test:.4f} MAE={mae_test:.4f} MAPE={mape_test:.2f}%   | "
          f"infer={infer_time_test_s:.2f}s")

    return {
        "rmse_train": rmse_train, "mae_train": mae_train, "mape_train": mape_train,
        "rmse_test": rmse_test,   "mae_test": mae_test,   "mape_test": mape_test,
        "train_time_s": train_time_s,
        "infer_time_train_s": infer_time_train_s,
        "infer_time_test_s": infer_time_test_s
    }

class ParametricLSTMCell_sigmoid(LSTMCell):
    def build(self, input_shape):
        super().build(input_shape)
        #Добавляем параметры активации (α и β для сигмоиды)
        self.alpha = self.add_weight(
            name='alpha',
            shape=(1,),
            initializer=Constant(1.0),
            trainable=True
        )
        self.beta = self.add_weight(
            name='beta',
            shape=(1,),
            initializer=Constant(0.0),
            trainable=True
        )

    def param_sigmoid(self, x):
        return 1.0 / (1.0 + K.exp(-self.alpha * (x - self.beta)))

    def call(self, inputs, states, training=None):
        h_tm1 = states[0]  #предыдущий скрытый
        c_tm1 = states[1]  #предыдущий cell state

        z = K.dot(inputs, self.kernel)
        z += K.dot(h_tm1, self.recurrent_kernel)
        if self.use_bias:
            z = K.bias_add(z, self.bias)

        z0, z1, z2, z3 = tf.split(z, num_or_size_splits=4, axis=1)

        #Используем нашу сигмоиду
        i = self.param_sigmoid(z0)     #input gate
        f = self.param_sigmoid(z1)     #forget gate
        c = f * c_tm1 + i * K.tanh(z2) #cell state
        o = self.param_sigmoid(z3)     #output gate

        h = o * K.tanh(c)
        return h, [h, c]

In [13]:
#@title (Опционально) Установка и настройка MLflow. Хоть сам MLflow пока не открывается, все равно запускаем эту ячейку, чтобы нужные данные сохранялись
USE_MLFLOW = True  #False, если не нужен MLflow-лог

if USE_MLFLOW:
    !pip -q install yfinance mlflow-skinny
    import mlflow, os
    MLFLOW_DIR = f"{BASE_DIR}/mlruns"
    os.makedirs(MLFLOW_DIR, exist_ok=True)
    mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")
    mlflow.set_experiment("fuzzy_vs_lstm")
    print("MLflow tracking:", mlflow.get_tracking_uri())

MLflow tracking: file:///content/drive/MyDrive/Fuzzy+Indicators_tests/mlruns


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [14]:
#@title Адаптеры
def fuzzy_basic_run(params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Вызывает твою FuzzyBasic(step, unit, companies, lookback, window)
    и возвращает словарь с метриками и путями к файлам.
    """
    step      = params["step"]
    units     = params["units"]
    ticker    = params["companies"]
    lookback  = params["lookback"]
    window    = params["window"]
    seed      = params.get("seed", 42)

    info = FuzzyBasic(step=step, unit=units, companies=ticker, lookback=lookback, window=window)

    pred_name = f"FuzzyBasic_{ticker}_{step}_lb{lookback}_u{units}_w{window}.csv"
    plot_name = f"FuzzyBasic_{ticker}_{step}_lb{lookback}_u{units}_w{window}.png"
    loss_name = f"FuzzyBasic_{ticker}_{step}_loss.png"

    info.update({
        "model": "FuzzyBasic",
        "ticker": ticker,
        "step": step,
        "lookback": lookback,
        "units": units,
        "window": window,
        "seed": seed,
        "preds_csv": f"{BASE_DIR}/preds/{pred_name}",
        "plot_png":  f"{BASE_DIR}/plots/{plot_name}",
        "loss_png":  f"{BASE_DIR}/plots/{loss_name}",
    })
    return info


def fuzzy_momentum_run(params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Вызывает твою FuzzyMomentum(step, unit, companies, lookback, window)
    и возвращает словарь с метриками и путям к файлам.
    """
    step      = params["step"]
    units     = params["units"]
    ticker    = params["companies"]
    lookback  = params["lookback"]
    window    = params["window"]
    seed      = params.get("seed", 42)

    info = FuzzyMomentum(step=step, unit=units, companies=ticker, lookback=lookback, window=window)

    pred_name = f"FuzzyMomentum_{ticker}_{step}_lb{lookback}_u{units}_w{window}.csv"
    plot_name = f"FuzzyMomentum_{ticker}_{step}_lb{lookback}_u{units}_w{window}.png"
    loss_name = f"FuzzyMomentum_{ticker}_{step}_loss.png"

    info.update({
        "model": "FuzzyMomentum",
        "ticker": ticker,
        "step": step,
        "lookback": lookback,
        "units": units,
        "window": window,
        "seed": seed,
        "preds_csv": f"{BASE_DIR}/preds/{pred_name}",
        "plot_png":  f"{BASE_DIR}/plots/{plot_name}",
        "loss_png":  f"{BASE_DIR}/plots/{loss_name}",
    })
    return info

def fuzzy_trend_run(params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Вызывает твою FuzzyTrend(step, unit, companies, lookback, window)
    и возвращает словарь с метриками и путям к файлам.
    """
    step      = params["step"]
    units     = params["units"]
    ticker    = params["companies"]
    lookback  = params["lookback"]
    window    = params["window"]
    seed      = params.get("seed", 42)

    info = FuzzyTrend(step=step, unit=units, companies=ticker, lookback=lookback, window=window)

    pred_name = f"FuzzyTrend_{ticker}_{step}_lb{lookback}_u{units}_w{window}.csv"
    plot_name = f"FuzzyTrend_{ticker}_{step}_lb{lookback}_u{units}_w{window}.png"
    loss_name = f"FuzzyTrend_{ticker}_{step}_loss.png"

    info.update({
        "model": "FuzzyTrend",
        "ticker": ticker,
        "step": step,
        "lookback": lookback,
        "units": units,
        "window": window,
        "seed": seed,
        "preds_csv": f"{BASE_DIR}/preds/{pred_name}",
        "plot_png":  f"{BASE_DIR}/plots/{plot_name}",
        "loss_png":  f"{BASE_DIR}/plots/{loss_name}",
    })
    return info

def fuzzy_volatility_run(params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Вызывает твою FuzzyVolatility(step, unit, companies, lookback, window)
    и возвращает словарь с метриками и путям к файлам.
    """
    step      = params["step"]
    units     = params["units"]
    ticker    = params["companies"]
    lookback  = params["lookback"]
    window    = params["window"]
    seed      = params.get("seed", 42)

    info = FuzzyVolatility(step=step, unit=units, companies=ticker, lookback=lookback, window=window)

    pred_name = f"FuzzyVolatility_{ticker}_{step}_lb{lookback}_u{units}_w{window}.csv"
    plot_name = f"FuzzyVolatility_{ticker}_{step}_lb{lookback}_u{units}_w{window}.png"
    loss_name = f"FuzzyVolatility_{ticker}_{step}_loss.png"

    info.update({
        "model": "FuzzyVolatility",
        "ticker": ticker,
        "step": step,
        "lookback": lookback,
        "units": units,
        "window": window,
        "seed": seed,
        "preds_csv": f"{BASE_DIR}/preds/{pred_name}",
        "plot_png":  f"{BASE_DIR}/plots/{plot_name}",
        "loss_png":  f"{BASE_DIR}/plots/{loss_name}",
    })
    return info

def fuzzy_all_run(params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Вызывает твою FuzzyAll(step, unit, companies, lookback, window)
    и возвращает словарь с метриками и путям к файлам.
    """
    step      = params["step"]
    units     = params["units"]
    ticker    = params["companies"]
    lookback  = params["lookback"]
    window    = params["window"]
    seed      = params.get("seed", 42)

    info = FuzzyAll(step=step, unit=units, companies=ticker, lookback=lookback, window=window)

    pred_name = f"FuzzyAll_{ticker}_{step}_lb{lookback}_u{units}_w{window}.csv"
    plot_name = f"FuzzyAll_{ticker}_{step}_lb{lookback}_u{units}_w{window}.png"
    loss_name = f"FuzzyAll_{ticker}_{step}_loss.png"

    info.update({
        "model": "FuzzyAll",
        "ticker": ticker,
        "step": step,
        "lookback": lookback,
        "units": units,
        "window": window,
        "seed": seed,
        "preds_csv": f"{BASE_DIR}/preds/{pred_name}",
        "plot_png":  f"{BASE_DIR}/plots/{plot_name}",
        "loss_png":  f"{BASE_DIR}/plots/{loss_name}",
    })
    return info


MODEL_REGISTRY = {
    "FuzzyBasic": fuzzy_basic_run,
    "FuzzyMomentum": fuzzy_momentum_run,
    "FuzzyTrend": fuzzy_trend_run,
    "FuzzyVolatility": fuzzy_volatility_run,
    "FuzzyAll": fuzzy_all_run,
}

In [15]:
#@title Сохранение артефактов (CSV/PNG) + реестр

def save_plot(fig, fname):
    fig.savefig(os.path.join(f"{BASE_DIR}/plots", fname), bbox_inches="tight")
    import matplotlib.pyplot as plt
    plt.close(fig)

def save_preds(df_preds, fname):
    df_preds.to_csv(os.path.join(f"{BASE_DIR}/preds", fname), index=False)

# --- имя файла предсказаний + проверка "уже сделано?" ---
def result_pred_path(model, ticker, step, units, lookback, seed, window=None):
    base = f"{BASE_DIR}/preds/{model}-{ticker.replace('=','').replace('^','')}-{step}-u{units}-lb{lookback}-s{seed}"
    if model == "FuzzyLSTM" and window is not None:
        base += f"-w{window}"
    return base + ".csv"

def already_done(model, ticker, step, units, lookback, seed, window=None):
    return os.path.exists(result_pred_path(model, ticker, step, units, lookback, seed, window))

In [16]:
#@title Функция registry_append

#--- безопасная дозапись в реестр (без FutureWarning) ---
def registry_append(row: dict):
    out = {k: row.get(k, "") for k in REGISTRY_COLS}

    id_cols = ["model", "ticker", "step", "lookback", "units", "window", "seed"]
    current_id_values = {col: out[col] for col in id_cols}

    if os.path.exists(REGISTRY_CSV) and os.stat(REGISTRY_CSV).st_size > 0:
        df_registry = pd.read_csv(REGISTRY_CSV)
    else:
        df_registry = pd.DataFrame(columns=REGISTRY_COLS)

    for col in id_cols:
        if col in df_registry.columns:
            df_registry[col] = df_registry[col].astype(str)
        current_id_values[col] = str(current_id_values[col])

    match_mask = pd.Series(True, index=df_registry.index)
    for col, value in current_id_values.items():
        if col in df_registry.columns:
            match_mask &= (df_registry[col] == value)
        else:
            match_mask = pd.Series(False, index=df_registry.index)
            break

    if match_mask.any():
        idx_to_update = df_registry[match_mask].index[0]
        for k, v in out.items():
            df_registry.loc[idx_to_update, k] = v
    else:
        df_registry = pd.concat([df_registry, pd.DataFrame([out], columns=REGISTRY_COLS)], ignore_index=True)

    df_registry.to_csv(REGISTRY_CSV, index=False)

In [17]:
#@title Конфигурация минимального пайплайна (5 модели × 25 инструментов × 3 шага)
#Категории (по 5 инструментов)
COMPANIES = {
    "stocks":      ["AAPL", "INTC", "TSLA", "NVDA", "MSFT"],
    "fx":          ["EURUSD=X", "CHF=X", "JPY=X", "GBPUSD=X", "AUDUSD=X"],
    "crypto":      ["BTC-USD", "ETH-USD", "SOL-USD", "BNB-USD", "XRP-USD"],
    "indices":     ["^GSPC", "^IXIC", "^DJI", "^FTSE", "^N225"],
    "commodities": ["GC=F", "CL=F", "SI=F", "HG=F", "NG=F"],
}
#Плоский список 10 тикеров:
TICKERS = [t for pair in COMPANIES.values() for t in pair]

#Три шага (интервалы) — как в твоём дереве:
STEPS = ["1m", "1h", "1d"]

#Фиксированные параметры из дерева:
UNITS_BY_STEP    = {"1m": 16, "1h": 32, "1d": 16}
LOOKBACK_BY_STEP = {"1m": 2, "1h": 10, "1d": 1}

#Для Fuzzy LSTM
FUZZY_WINDOW_DEFAULT = 10

#Seed для одинаковой воспроизводимости с любого компьютера
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

In [18]:
#@title Раннер для одного теста.
def run_one(model_name: str, params: Dict[str, Any], use_mlflow: bool = True):
    run_fn = MODEL_REGISTRY[model_name]
    info = run_fn(params)  #<- вызывает функцию и возвращает метрики + пути к файлам

    #записываем в registry.csv
    registry_append({
        "model": info["model"], "ticker": info["ticker"], "step": info["step"],
        "lookback": info["lookback"], "units": info["units"], "window": info["window"], "seed": info["seed"],
        "rmse_train": info["rmse_train"], "mae_train": info["mae_train"], "mape_train": info["mape_train"],
        "rmse_test":  info["rmse_test"],  "mae_test":  info["mae_test"],  "mape_test":  info["mape_test"],
        "train_time_s": info["train_time_s"],
        "infer_time_train_s": info.get("infer_time_train_s", None),
        "infer_time_test_s":  info.get("infer_time_test_s",  None),
        "ts": time.time()
    })

    #(опционально) лог в MLflow-skinny
    if USE_MLFLOW:
        import mlflow
        with mlflow.start_run(run_name=f"{info['model']}-{info['ticker']}-{info['step']}"):
            mlflow.log_params({
                "model": info["model"], "ticker": info["ticker"], "step": info["step"],
                "lookback": info["lookback"], "units": info["units"], "window": info["window"], "seed": info["seed"]
            })
            mlflow.log_metrics({
                "RMSE_TRAIN": info["rmse_train"], "MAE_TRAIN": info["mae_train"], "MAPE_TRAIN": info["mape_train"],
                "RMSE_TEST":  info["rmse_test"],  "MAE_TEST":  info["mae_test"],  "MAPE_TEST":  info["mape_test"],
                "train_time_s": info["train_time_s"]
            })
            #прикрепляем артефакты
            for p in [info["preds_csv"], info["plot_png"], info["loss_png"]]:
                if os.path.exists(p): mlflow.log_artifact(p)

    return info

In [19]:
#@title Так как один тест может долго работать, а колаб при долгой работе ячейки может оключиться, то можем запускать и по одному тесту
params = {
    "step": "1d",
    "companies": "NG=F",
    "units": 16,
    "lookback": 1,
    "window": 10,
    "seed": 42,
}

In [21]:
info1 = run_one("FuzzyBasic", params, use_mlflow=USE_MLFLOW)

TRAIN: RMSE=0.1886 MAE=0.1175 MAPE=2.42%  | time=79.95s, infer=0.80s
TEST : RMSE=0.2233 MAE=0.1302 MAPE=3.53%   | infer=0.19s


In [22]:
info2 = run_one("FuzzyMomentum", params, use_mlflow=USE_MLFLOW)

TRAIN: RMSE=0.1829 MAE=0.1148 MAPE=2.40%  | time=77.55s, infer=0.84s
TEST : RMSE=0.2248 MAE=0.1311 MAPE=3.54%   | infer=0.17s


In [23]:
info3 = run_one("FuzzyTrend", params, use_mlflow=USE_MLFLOW)

TRAIN: RMSE=0.1886 MAE=0.1175 MAPE=2.43%  | time=80.58s, infer=0.81s
TEST : RMSE=0.2240 MAE=0.1305 MAPE=3.53%   | infer=0.16s


In [24]:
info4 = run_one("FuzzyVolatility", params, use_mlflow=USE_MLFLOW)

TRAIN: RMSE=0.1893 MAE=0.1181 MAPE=2.43%  | time=76.27s, infer=0.82s
TEST : RMSE=0.2232 MAE=0.1309 MAPE=3.54%   | infer=0.17s


In [25]:
info5 = run_one("FuzzyAll", params, use_mlflow=USE_MLFLOW)

TRAIN: RMSE=0.1822 MAE=0.1146 MAPE=2.40%  | time=74.80s, infer=0.83s
TEST : RMSE=0.2244 MAE=0.1310 MAPE=3.53%   | infer=0.17s
